# Image Perceptual Losses Comparison Example

This notebook provides an example of using existing perceptual loss functions as objectives when optimizing lighting, as discussed in [section 3.1](https://scholarsarchive.byu.edu/cgi/viewcontent.cgi?article=12256&context=etd#page=19.1) of the thesis. A loop is configured to easily compare the effect of each of these criteria on the selected scene:
- SSIM (`SSIMLoss`)
- LPIPS (`LPIPSLoss`)
- VGG Style Transfer, with both VGG-16 and VGG-19 backbones (`VGGStyleTransferLoss`)

In [ ]:
import os
import sys

if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from examples.example_scenes import BlenderManScene, CandleScene, CarScene, CarStudioScene, DinoScene, EinarScene, EinarSmallDomeScene, FlowerPotScene, HouseScene, RedCarScene, SciFiRobotScene, SpringPortraitScene, SpringPortraitSmallDomeScene, SpringScene
from losses.image_image import (
    LPIPSLoss,
    SSIMLoss,
    VGGStyleTransferLoss,
)
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.optimize import optimize_with_criterion
from utils.record_keeping.experiment import FolderManager


In [ ]:
# Define scenes to benchmark
scenes_to_test = [
    SciFiRobotScene(device=device),
    CarScene(device=device),
]


In [ ]:
# Select target reference image (uncomment desired image)
target_image_path = "examples/EXAMPLE_REFERENCE_IMAGES/golden_hour_01.jpg"
# target_image_path = "examples/EXAMPLE_REFERENCE_IMAGES/golden_hour_02.jpg"
# target_image_path = "examples/EXAMPLE_REFERENCE_IMAGES/sunset.jpg"
# target_image_path = "examples/EXAMPLE_REFERENCE_IMAGES/cemrecan-yurtman-rx5OLYpKViI-unsplash.jpg"
# target_image_path = "examples/EXAMPLE_REFERENCE_IMAGES/jahanzeb-ahsan-_cM4CsbDhwQ-unsplash.jpg"

In [ ]:
# Hyperparameters
lr = 0.06
n_iter = 250
global_seed = 2
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())

# Uncomment the loss functions you want to test here:
loss_configs = [
    # ("SSIM", lambda: SSIMLoss(reference_image=target_image_path, device=device)),
    # ("LPIPS", lambda: LPIPSLoss(reference_image=target_image_path, device=device)),
    ("VGG-16 Style", lambda: VGGStyleTransferLoss(reference_image=target_image_path, backbone="vgg16", device=device)),
    # ("VGG-19 Style", lambda: VGGStyleTransferLoss(reference_image=target_image_path, backbone="vgg19", device=device)),
]

# Optimization Benchmark Loop
output_directory = "image_perceptual_example"

for scene in scenes_to_test:
    for loss_name, criterion_fn in loss_configs:
        print(f"--- Running {loss_name} on {scene.name} ---")
        criterion = criterion_fn()

        optimize_with_criterion(
            scene,
            lr,
            n_iter,
            criterion,
            starting_multiplier_std=(0.1, 0.1, 0.1),
            output_subdirectory_name=output_directory,
            n_results=1,
            render_color_space_converter=color_space_converter,
            require_physically_plausible_multipliers=True,
            title_prefix=f"{loss_name} ({scene.name})",
            device=device,
            save_every=50,
            model_name=loss_name,
            pretrained_source="",
            seed=global_seed,
        )
